# Online Retail EDA

## Exploratory Data Analysis for Data Engineering Portfolio

This notebook explores the cleaned Online Retail dataset generated by the ingestion stage of the pipeline.

The goal is to validate data quality, understand the dataset, and identify business insights before building the staging and data warehouse layers.

## Dataset Used

`data/processed/cleaned_sales.csv`

## Main Objectives

1. Load the cleaned dataset
2. Review structure and data types
3. Validate missing values and duplicates
4. Analyze sales behavior
5. Identify top products, customers, and countries
6. Generate portfolio-ready insights


## 1. Import Libraries

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)


## 2. Define Project Paths

The notebook uses relative paths based on the project root structure.

Expected structure:

```text
OnlineRetail/
├── data/
│   ├── raw/
│   └── processed/
├── output/
├── db/
├── logs/
├── notebooks/
├── scripts/
├── run_pipeline.py
└── run_pipeline.sh
```


In [ ]:
# If this notebook is inside the notebooks/ folder, parent.parent points to the project root
BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

DATA_PATH = BASE_DIR / "data" / "processed" / "cleaned_sales.csv"

print("Project root:", BASE_DIR)
print("Dataset path:", DATA_PATH)


## 3. Load Cleaned Dataset

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")

df.head()


## 4. Dataset Structure

In [ ]:
df.info()


In [ ]:
df.dtypes


## 5. Basic Data Quality Checks

In [ ]:
quality_summary = pd.DataFrame({
    "column": df.columns,
    "missing_values": df.isna().sum().values,
    "missing_percentage": (df.isna().mean().values * 100).round(2),
    "unique_values": df.nunique(dropna=False).values,
    "data_type": df.dtypes.astype(str).values
})

quality_summary


In [ ]:
duplicate_rows = df.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_rows:,}")


## 6. Convert Date Column

This ensures the invoice date is treated as a timestamp for time-based analysis.


In [ ]:
df["invoicedate"] = pd.to_datetime(df["invoicedate"], errors="coerce")

print("Invalid invoice dates:", df["invoicedate"].isna().sum())
print("Date range:")
print("Min date:", df["invoicedate"].min())
print("Max date:", df["invoicedate"].max())


## 7. Numeric Field Validation

In [ ]:
numeric_validation = {
    "negative_quantity_rows": int((df["quantity"] < 0).sum()),
    "zero_quantity_rows": int((df["quantity"] == 0).sum()),
    "negative_unitprice_rows": int((df["unitprice"] < 0).sum()),
    "zero_unitprice_rows": int((df["unitprice"] == 0).sum())
}

numeric_validation


## 8. Create Analytical Columns for EDA

This notebook can create temporary analytical columns for exploration.  
These columns are not necessarily part of the final cleaned dataset.


In [ ]:
df["sales_amount"] = df["quantity"] * df["unitprice"]
df["invoice_date"] = df["invoicedate"].dt.date
df["year_month"] = df["invoicedate"].dt.to_period("M").astype(str)

df[["quantity", "unitprice", "sales_amount", "invoice_date", "year_month"]].head()


## 9. General Business KPIs

In [ ]:
total_revenue = df["sales_amount"].sum()
total_transactions = df["invoiceno"].nunique()
total_products = df["stockcode"].nunique()
total_customers = df["customerid"].nunique()
total_countries = df["country"].nunique()

kpi_summary = pd.DataFrame({
    "metric": [
        "Total Revenue",
        "Unique Invoices",
        "Unique Products",
        "Unique Customers",
        "Unique Countries"
    ],
    "value": [
        round(total_revenue, 2),
        total_transactions,
        total_products,
        total_customers,
        total_countries
    ]
})

kpi_summary


## 10. Monthly Sales Trend

In [ ]:
monthly_sales = (
    df.groupby("year_month", as_index=False)
      .agg(total_sales=("sales_amount", "sum"),
           total_orders=("invoiceno", "nunique"))
      .sort_values("year_month")
)

monthly_sales.head()


In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(monthly_sales["year_month"], monthly_sales["total_sales"], marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Year-Month")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 11. Top 10 Countries by Sales

In [ ]:
top_countries = (
    df.groupby("country", as_index=False)
      .agg(total_sales=("sales_amount", "sum"),
           total_orders=("invoiceno", "nunique"))
      .sort_values("total_sales", ascending=False)
      .head(10)
)

top_countries


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(top_countries["country"], top_countries["total_sales"])
plt.title("Top 10 Countries by Sales")
plt.xlabel("Country")
plt.ylabel("Total Sales")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


## 12. Top 10 Products by Sales

In [ ]:
top_products = (
    df.groupby(["stockcode", "description"], as_index=False)
      .agg(total_sales=("sales_amount", "sum"),
           total_quantity=("quantity", "sum"))
      .sort_values("total_sales", ascending=False)
      .head(10)
)

top_products


In [ ]:
plt.figure(figsize=(12, 5))
plt.bar(top_products["description"], top_products["total_sales"])
plt.title("Top 10 Products by Sales")
plt.xlabel("Product")
plt.ylabel("Total Sales")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()


## 13. Top 10 Customers by Sales

In [ ]:
top_customers = (
    df[df["customerid"] != "N/A"]
    .groupby("customerid", as_index=False)
    .agg(total_sales=("sales_amount", "sum"),
         total_orders=("invoiceno", "nunique"))
    .sort_values("total_sales", ascending=False)
    .head(10)
)

top_customers


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(top_customers["customerid"], top_customers["total_sales"])
plt.title("Top 10 Customers by Sales")
plt.xlabel("Customer ID")
plt.ylabel("Total Sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 14. Sales Distribution

This helps identify extreme values and possible outliers.


In [ ]:
df["sales_amount"].describe()


In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df["sales_amount"], bins=50)
plt.title("Sales Amount Distribution")
plt.xlabel("Sales Amount")
plt.ylabel("Frequency")
plt.tight_layout()
plt.show()


## 15. Non-Commercial or Special Records Review

This section helps validate whether records such as discounts, adjustments, samples, or postage are present before the transformation layer filters them.


In [ ]:
keywords = [
    "POSTAGE", "TEST", "SAMPLE", "ADJUST", "DISCOUNT",
    "CHARGES", "CARRIAGE", "GIFT", "MANUAL",
    "UNKNOWN", "CHECK", "DAMAGED"
]

pattern = "|".join(keywords)

special_records = df[
    df["description"].astype(str).str.upper().str.contains(pattern, na=False)
]

print(f"Special records found: {len(special_records):,}")
special_records.head(10)


## 16. EDA Summary

Key findings to complete after running the notebook:

- Total revenue:
- Top country by sales:
- Top product by sales:
- Top customer by sales:
- Months with highest sales:
- Data quality issues identified:
- Records that should be excluded from the staging layer:

## Portfolio Note

This notebook supports the data pipeline by validating the cleaned dataset before the transformation and data warehouse stages.
